# 03 — Data Preprocessing

## Objective

Convert the raw FORCE 2020 well-log data into a consistent,
ML-ready representation while preserving the geological structure
of the wells.

### Input
Raw LAS files from `data/raw/FORCE_2020/`

### Main operations
1. Load raw well logs
2. Select required curves
3. Convert invalid values to NaN
4. Remove samples without lithology labels for supervised training
5. Handle missing log values
6. Apply physically motivated transformations
7. Preserve well and depth information
8. Save preprocessing decisions/results for the next stage

### Important rule

All preprocessing parameters that are learned from data
(e.g. median imputation values) must be calculated from the
training wells only and later applied unchanged to validation
and test wells.

In [33]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import lasio
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

In [14]:
# pip install tqdm

setting the paths for input and output of of the processing files

In [15]:
PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "FORCE_2020"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data:", RAW_DIR)
print("Interim:", INTERIM_DIR)
print("Processed:", PROCESSED_DIR)

Raw data: ../data/raw/FORCE_2020
Interim: ../data/interim
Processed: ../data/processed


defining the curves .... prelimnary curvers for the preprocessing 

In [16]:
INPUT_CURVES = [
    "GR",
    "RDEP",
    "RMED",
    "DTC",
    "RHOB",
]

LABEL_CURVE = "FORCE_2020_LITHOFACIES_LITHOLOGY"

DEPTH_CURVE = "DEPT"

print("Input curves:", INPUT_CURVES)
print("Label:", LABEL_CURVE)
print("Depth:", DEPTH_CURVE)

Input curves: ['GR', 'RDEP', 'RMED', 'DTC', 'RHOB']
Label: FORCE_2020_LITHOFACIES_LITHOLOGY
Depth: DEPT


the 12 lithology mapping provided by the force 2020 competetion database 

In [17]:
LITHOLOGY_MAP = {
    30000: "Sandstone",
    65000: "Shale",
    65030: "Sandstone/Shale",
    70000: "Limestone",
    70032: "Chalk",
    74000: "Dolomite",
    80000: "Marl",
    86000: "Anhydrite",
    88000: "Halite",
    90000: "Coal",
    93000: "Basement",
    99000: "Tuff",
}

firdt we will se the one well

In [20]:
las_files = sorted(RAW_DIR.glob("*.[lL][aA][sS]"))

print("LAS files found:", len(las_files))
print(las_files[:5])

LAS files found: 118
[PosixPath('../data/raw/FORCE_2020/15_9-13.las'), PosixPath('../data/raw/FORCE_2020/15_9-14.las'), PosixPath('../data/raw/FORCE_2020/15_9-15.las'), PosixPath('../data/raw/FORCE_2020/15_9-17.las'), PosixPath('../data/raw/FORCE_2020/15_9-23.las')]


In [21]:
las = lasio.read(las_files[0])
df = las.df().reset_index()

df.head()

,DEPT,FORCE_2020_LITHOFACIES_CONFIDENCE,FORCE_2020_LITHOFACIES_LITHOLOGY,CALI,MUDWEIGHT,ROP,RDEP,RSHA,RMED,RXO,...,DTC,NPHI,PEF,GR,RHOB,DRHO,DEPTH_MD,X_LOC,Y_LOC,Z_LOC
0,25.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,206.224609,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,25.152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,206.224625,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,25.304,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,206.224655,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,25.456,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,206.224670,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,25.608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,206.224701,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


trying to make temperory well split for applying the decisons and result from previous notebooks on the dataset

In [23]:

SEED = 42

las_files = sorted(RAW_DIR.glob("*.[lL][aA][sS]"))

well_names = [p.stem for p in las_files]


In [25]:
rng = np.random.default_rng(SEED)

shuffled_wells = well_names.copy()
rng.shuffle(shuffled_wells)

n_train = int(len(shuffled_wells) * 0.8)

train_wells = shuffled_wells[:n_train]
validation_wells = shuffled_wells[n_train:]

print("Training wells:", len(train_wells))
print("Validation wells:", len(validation_wells))

Training wells: 94
Validation wells: 24


In [ ]:
print("Overlap:", set(train_wells) & set(validation_wells))#well tell no overlap of split as set()

Overlap: set()


In [27]:
split_df = pd.DataFrame({
    "WELL": well_names,
    "SPLIT": [
        "train" if w in train_wells else "validation"
        for w in well_names
    ]
})

split_df.to_csv(
    INTERIM_DIR / "temporary_well_split.csv",
    index=False
)

split_df["SPLIT"].value_counts()

SPLIT
train         94
validation    24
Name: count, dtype: int64

creating a preprocessing function structure for the single well to understand the characteristics of the functions

In [28]:
INPUT_CURVES = ["GR", "RDEP", "RMED", "DTC", "RHOB"]
LABEL_CURVE = "FORCE_2020_LITHOFACIES_LITHOLOGY"

LITHOLOGY_CODES = {
    30000, 65000, 65030, 70000, 70032, 74000,
    80000, 86000, 88000, 90000, 93000, 99000
}

In [42]:
def preprocess_well(df):
    df = df.copy()

    # Normalize column names
    df.columns = [str(c).strip().upper() for c in df.columns]

    # Keep required columns
    required = INPUT_CURVES + [LABEL_CURVE]

    # missing = [c for c in required if c not in df.columns]
    # # if missing:
    # #     raise ValueError(f"Missing required curves: {missing}")

    # if missing:
    #     print(f"Missing curves: {missing}")

    # df = df[required].copy()


    missing = [c for c in required if c not in df.columns]

    if missing:
        print(f"Missing curves: {missing}")
        for c in missing:
            df[c] = np.nan

    df = df[required].copy()

    # Invalid values identified in 02_data_quality
    df.loc[df["DTC"] <= 0, "DTC"] = np.nan
    df.loc[df["RHOB"] <= 0, "RHOB"] = np.nan
    df.loc[df["GR"] < 0, "GR"] = np.nan

    # Resistivity must be positive before log10
    df.loc[df["RDEP"] <= 0, "RDEP"] = np.nan
    df.loc[df["RMED"] <= 0, "RMED"] = np.nan

    # Keep only known lithology labels
    df[LABEL_CURVE] = pd.to_numeric(
        df[LABEL_CURVE],
        errors="coerce"
    )

    df.loc[
        ~df[LABEL_CURVE].isin(LITHOLOGY_CODES),
        LABEL_CURVE
    ] = np.nan

    return df

now testing on a well

In [43]:
las = lasio.read(las_files[0])
df = las.df().reset_index()

processed = preprocess_well(df)

print("Original rows:", len(df))
print("Processed rows:", len(processed))

processed.head()

Original rows: 21441
Processed rows: 21441


,GR,RDEP,RMED,DTC,RHOB,FORCE_2020_LITHOFACIES_LITHOLOGY
0,NaN,NaN,NaN,206.224609,NaN,NaN
1,NaN,NaN,NaN,206.224625,NaN,NaN
2,NaN,NaN,NaN,206.224655,NaN,NaN
3,NaN,NaN,NaN,206.224670,NaN,NaN
4,NaN,NaN,NaN,206.224701,NaN,NaN


saving the result

In [44]:

output_folder = "./visualization_result"
output_file = "one well preprocessing.csv"

os.makedirs(output_folder, exist_ok=True)

full_path = os.path.join(output_folder, output_file)

processed.to_csv(full_path, index=False)

print(f"Saved successfully to: {full_path}")

Saved successfully to: ./visualization_result/one well preprocessing.csv


now for all the wells

In [45]:
processed_wells = {}

for las_path in tqdm(las_files):

    las = lasio.read(las_path)
    df = las.df().reset_index()

    processed = preprocess_well(df)

    well_name = las_path.stem
    processed["WELL"] = well_name

    processed_wells[well_name] = processed

print("Processed wells:", len(processed_wells))

  0%|          | 0/118 [00:00<?, ?it/s]

Missing curves: ['RMED']
Processed wells: 118


saving this all 118 well preprocessing data as "all well preprocessing.csv"

In [46]:

output_folder = "./visualization_result"
output_file = "all well preprocessing.csv"

os.makedirs(output_folder, exist_ok=True)

full_path = os.path.join(output_folder, output_file)

processed.to_csv(full_path, index=False)

print(f"Saved successfully to: {full_path}")

Saved successfully to: ./visualization_result/all well preprocessing.csv


check missing values

In [47]:
all_processed = pd.concat(
    processed_wells.values(),
    ignore_index=True
)

missing_summary = pd.DataFrame({
    "missing_count": all_processed[INPUT_CURVES].isna().sum(),
    "missing_percent": (
        all_processed[INPUT_CURVES].isna().mean() * 100
    )
})

missing_summary

,missing_count,missing_percent
GR,42244,1.807122
RDEP,118446,5.066907
RMED,244772,10.470907
DTC,552970,23.655064
RHOB,963032,41.196780


now missing values by well and saving

In [48]:
well_missing = []

for well_name, df in processed_wells.items():
    for curve in INPUT_CURVES:
        well_missing.append({
            "WELL": well_name,
            "CURVE": curve,
            "MISSING_PERCENT": df[curve].isna().mean() * 100
        })

well_missing_df = pd.DataFrame(well_missing)

well_missing_df.head()

,WELL,CURVE,MISSING_PERCENT
0,15_9-13,GR,2.523203
1,15_9-13,RDEP,2.262021
2,15_9-13,RMED,2.257357
3,15_9-13,DTC,0.741570
4,15_9-13,RHOB,14.439625


In [49]:

output_folder = "./visualization_result"
output_file = "missing values by well.csv"

os.makedirs(output_folder, exist_ok=True)

full_path = os.path.join(output_folder, output_file)

processed.to_csv(full_path, index=False)

print(f"Saved successfully to: {full_path}")

Saved successfully to: ./visualization_result/missing values by well.csv


In [50]:
well_missing_df.pivot(
    index="WELL",
    columns="CURVE",
    values="MISSING_PERCENT"
).describe()

CURVE,DTC,GR,RDEP,RHOB,RMED
count,118.000000,118.000000,118.000000,118.000000,118.000000
mean,23.778348,2.059359,5.008061,40.448447,10.338383
std,24.396988,4.727418,10.601473,27.136481,16.733764
min,0.642048,0.005497,0.003837,0.010281,0.000000
25%,4.383376,0.257625,0.240035,18.269269,0.824209
50%,16.244124,0.609348,1.786972,38.183229,3.537389
75%,33.087845,1.926044,3.817855,60.367532,14.144552
max,90.745157,39.046913,80.908865,91.744103,100.000000


In [51]:
well_missing_df[
    well_missing_df["MISSING_PERCENT"] > 50
].sort_values(
    ["CURVE", "MISSING_PERCENT"],
    ascending=[True, False]
).head(30)

,WELL,CURVE,MISSING_PERCENT
543,35_8-6 S,DTC,90.745157
538,35_8-4,DTC,88.183518
248,30_3-5 S,DTC,86.868400
323,31_5-4 S,DTC,84.546961
588,7_1-2 S,DTC,84.412335
553,35_9-2,DTC,79.233689
388,34_11-1,DTC,78.918828
493,35_11-13,DTC,78.084628
498,35_11-15 S,DTC,74.150382
223,25_9-1,DTC,74.125961


Calculate the median of each curve from training wells only, then use those medians to fill missing values in both training and validation wells

In [52]:
# Combine wells while keeping the WELL column
all_processed = pd.concat(
    processed_wells.values(),
    ignore_index=True
)

# Identify training rows using our temporary well split
train_mask = all_processed["WELL"].isin(train_wells)
val_mask = all_processed["WELL"].isin(validation_wells)

# Calculate medians ONLY from training wells
train_medians = (
    all_processed.loc[train_mask, INPUT_CURVES]
    .median()
)

print("Training-set medians:")
print(train_medians)

Training-set medians:
GR       70.047695
RDEP      1.430475
RMED      1.383175
DTC     116.738678
RHOB      2.322635
dtype: float64


In [53]:
# Create a copy for testing the imputation strategy
imputed_data = all_processed.copy()

# Add missing-value indicators
for curve in INPUT_CURVES:
    imputed_data[f"{curve}_MISSING"] = (
        imputed_data[curve].isna().astype("int8")
    )

# Fill missing values using TRAINING medians only
for curve in INPUT_CURVES:
    imputed_data[curve] = (
        imputed_data[curve]
        .fillna(train_medians[curve])
    )

print("Remaining missing values:")
print(imputed_data[INPUT_CURVES].isna().sum())

Remaining missing values:
GR      0
RDEP    0
RMED    0
DTC     0
RHOB    0
dtype: int64


missing indicators

In [54]:
for curve in INPUT_CURVES:
    col = f"{curve}_MISSING"
    print(
        f"{col}:",
        imputed_data[col].sum(),
        "samples originally missing"
    )

GR_MISSING: 42244 samples originally missing
RDEP_MISSING: 118446 samples originally missing
RMED_MISSING: 244772 samples originally missing
DTC_MISSING: 552970 samples originally missing
RHOB_MISSING: 963032 samples originally missing


In [55]:
missing_by_well = (
    imputed_data
    .groupby("WELL")[[f"{c}_MISSING" for c in INPUT_CURVES]]
    .mean()
    * 100
)

missing_by_well.describe()

,GR_MISSING,RDEP_MISSING,RMED_MISSING,DTC_MISSING,RHOB_MISSING
count,118.000000,118.000000,118.000000,118.000000,118.000000
mean,2.059359,5.008061,10.338383,23.778348,40.448447
std,4.727418,10.601473,16.733764,24.396988,27.136481
min,0.005497,0.003837,0.000000,0.642048,0.010281
25%,0.257625,0.240035,0.824209,4.383376,18.269269
50%,0.609348,1.786972,3.537389,16.244124,38.183229
75%,1.926044,3.817855,14.144552,33.087845,60.367532
max,39.046913,80.908865,100.000000,90.745157,91.744103


In [56]:
well = "35_8-6 S"

test_well = imputed_data[
    imputed_data["WELL"] == well
].copy()

test_well[
    INPUT_CURVES + [f"{c}_MISSING" for c in INPUT_CURVES]
].head(20)

,GR,RDEP,RMED,DTC,RHOB,GR_MISSING,RDEP_MISSING,RMED_MISSING,DTC_MISSING,RHOB_MISSING
2142031,70.047695,1.430475,1.383175,116.738678,2.322635,1,1,1,1,1
2142032,70.047695,1.430475,1.383175,116.738678,2.322635,1,1,1,1,1
2142033,70.047695,1.430475,1.383175,116.738678,2.322635,1,1,1,1,1
2142034,70.047695,1.430475,1.383175,116.738678,2.322635,1,1,1,1,1
2142035,70.047695,1.430475,1.383175,116.738678,2.322635,1,1,1,1,1
2142036,70.047695,1.430475,1.383175,116.738678,2.322635,1,1,1,1,1
2142037,70.047695,1.430475,1.383175,116.738678,2.322635,1,1,1,1,1
2142038,70.047695,1.430475,1.383175,116.738678,2.322635,1,1,1,1,1
2142039,70.047695,1.430475,1.383175,116.738678,2.322635,1,1,1,1,1
2142040,70.047695,1.430475,1.383175,116.738678,2.322635,1,1,1,1,1


one well continuaous run


In [57]:
well = "35_8-6 S"

original_well = processed_wells[well].copy()

for curve in INPUT_CURVES:
    missing = original_well[curve].isna()

    # Identify consecutive missing runs
    groups = missing.ne(missing.shift()).cumsum()

    gaps = (
        original_well.loc[missing]
        .groupby(groups[missing])
        .size()
        .sort_values(ascending=False)
    )

    print(f"\n{curve}")
    print("Largest missing runs:")
    print(gaps.head(5).to_string())


GR
Largest missing runs:
GR
1    469
3     13

RDEP
Largest missing runs:
RDEP
1    643
3     31

RMED
Largest missing runs:
RMED
1    643
7    349
9    109
5     94
3     51

DTC
Largest missing runs:
DTC
1    21371
3      318

RHOB
Largest missing runs:
RHOB
1    9303
3     238
5     105


for all well

In [58]:
gap_summary = []

for well, df in processed_wells.items():

    for curve in INPUT_CURVES:

        missing = df[curve].isna()
        groups = missing.ne(missing.shift()).cumsum()

        runs = (
            df.loc[missing]
            .groupby(groups[missing])
            .size()
        )

        if len(runs) > 0:
            gap_summary.append({
                "WELL": well,
                "CURVE": curve,
                "MAX_GAP_SAMPLES": runs.max(),
                "MAX_GAP_METERS": runs.max() * 0.152
            })

gap_summary = pd.DataFrame(gap_summary)

gap_summary.groupby("CURVE")[
    "MAX_GAP_METERS"
].describe()

,count,mean,std,min,25%,50%,75%,max
CURVE,,,,,,,,
DTC,118.0,661.999932,767.508798,14.896,105.374,446.348,912.000,3528.224
GR,118.0,50.270780,108.061159,0.152,7.600,12.388,48.298,880.080
RDEP,118.0,144.512068,378.572389,0.152,5.662,51.528,97.508,3458.608
RHOB,118.0,1191.689017,967.780770,0.152,501.144,931.380,1676.826,3990.304
RMED,117.0,293.353504,552.257774,0.152,12.464,73.112,360.392,3722.480


- RHOB: median largest gap = 931 m
- DTC: median = 446 m
- RMED: median = 73 m
- RDEP: median = 52 m
- GR: median = 12 m

In [59]:
gap_summary.groupby("CURVE")["MAX_GAP_METERS"].quantile(
    [0.25, 0.50, 0.75, 0.90, 0.95]
)

CURVE      
DTC    0.25     105.3740
       0.50     446.3480
       0.75     912.0000
       0.90    1576.3464
       0.95    2261.8664
GR     0.25       7.6000
       0.50      12.3880
       0.75      48.2980
       0.90     102.9648
       0.95     228.0456
RDEP   0.25       5.6620
       0.50      51.5280
       0.75      97.5080
       0.90     303.1032
       0.95     669.9324
RHOB   0.25     501.1440
       0.50     931.3800
       0.75    1676.8260
       0.90    2746.2448
       0.95    3205.0872
RMED   0.25      12.4640
       0.50      73.1120
       0.75     360.3920
       0.90     871.0512
       0.95    1079.4128
Name: MAX_GAP_METERS, dtype: float64

In [60]:
# Create the clean transformed dataset
preprocessed_data = all_processed.copy()

# Log-transform resistivity
preprocessed_data["RDEP_LOG10"] = np.log10(preprocessed_data["RDEP"])
preprocessed_data["RMED_LOG10"] = np.log10(preprocessed_data["RMED"])

print(preprocessed_data[
    ["RDEP", "RDEP_LOG10", "RMED", "RMED_LOG10"]
].head())

   RDEP  RDEP_LOG10  RMED  RMED_LOG10
0   NaN         NaN   NaN         NaN
1   NaN         NaN   NaN         NaN
2   NaN         NaN   NaN         NaN
3   NaN         NaN   NaN         NaN
4   NaN         NaN   NaN         NaN


In [61]:
print(
    preprocessed_data[
        ["RDEP_LOG10", "RMED_LOG10"]
    ].isna().sum()
)

RDEP_LOG10    118446
RMED_LOG10    244772
dtype: int64


Let's define the actual candidate ML features:

In [62]:
MODEL_FEATURES = [
    "GR",
    "RDEP_LOG10",
    "RMED_LOG10",
    "DTC",
    "RHOB",
]

print(MODEL_FEATURES)

['GR', 'RDEP_LOG10', 'RMED_LOG10', 'DTC', 'RHOB']


In [63]:
preprocessed_data[MODEL_FEATURES].describe()

,GR,RDEP_LOG10,RMED_LOG10,DTC,RHOB
count,2.295395e+06,2.219193e+06,2.092867e+06,1.784669e+06,1.374607e+06
mean,7.160906e+01,2.156408e-01,2.013824e-01,1.173622e+02,2.284587e+00
std,3.642843e+01,4.070473e-01,3.751293e-01,3.083445e+01,2.536549e-01
min,0.000000e+00,-4.000000e+00,-4.000000e+00,9.369886e-02,3.211390e-03
25%,4.752789e+01,-3.184389e-02,-4.244077e-02,9.094017e+01,2.092561e+00
50%,6.890990e+01,1.502686e-01,1.379469e-01,1.161028e+02,2.320878e+00
75%,9.012186e+01,3.658463e-01,3.540228e-01,1.439608e+02,2.489749e+00
max,1.141292e+03,3.301029e+00,3.301019e+00,3.204789e+02,3.457820e+00


creating a baseline dataset with missing indicators, while keeping the actual missing values as nan

In [64]:
BASELINE_FEATURES = MODEL_FEATURES.copy()

for curve in MODEL_FEATURES:
    preprocessed_data[f"{curve}_MISSING"] = (
        preprocessed_data[curve].isna().astype("int8")
    )

print("Features:")
print(BASELINE_FEATURES)

print("\nMissing indicators:")
print([f"{c}_MISSING" for c in MODEL_FEATURES])

Features:
['GR', 'RDEP_LOG10', 'RMED_LOG10', 'DTC', 'RHOB']

Missing indicators:
['GR_MISSING', 'RDEP_LOG10_MISSING', 'RMED_LOG10_MISSING', 'DTC_MISSING', 'RHOB_MISSING']


In [65]:
preprocessed_data[
    BASELINE_FEATURES +
    [f"{c}_MISSING" for c in MODEL_FEATURES]
].head()

,GR,RDEP_LOG10,RMED_LOG10,DTC,RHOB,GR_MISSING,RDEP_LOG10_MISSING,RMED_LOG10_MISSING,DTC_MISSING,RHOB_MISSING
0,NaN,NaN,NaN,206.224609,NaN,1,1,1,0,1
1,NaN,NaN,NaN,206.224625,NaN,1,1,1,0,1
2,NaN,NaN,NaN,206.224655,NaN,1,1,1,0,1
3,NaN,NaN,NaN,206.224670,NaN,1,1,1,0,1
4,NaN,NaN,NaN,206.224701,NaN,1,1,1,0,1


In [66]:
# Check whether any row has all five log features missing
all_missing = preprocessed_data[MODEL_FEATURES].isna().all(axis=1)

print("Rows with all 5 features missing:", all_missing.sum())
print(
    "Percentage:",
    round(all_missing.mean() * 100, 3),
    "%"
)

Rows with all 5 features missing: 25865
Percentage: 1.106 %


In [67]:
valid_feature_rows = ~preprocessed_data[MODEL_FEATURES].isna().all(axis=1)

print("Total rows:", len(preprocessed_data))
print("Rows with at least one log:", valid_feature_rows.sum())
print("Rows removed:", (~valid_feature_rows).sum())
print(
    "Percentage removed:",
    round((~valid_feature_rows).mean() * 100, 3),
    "%"
)

Total rows: 2337639
Rows with at least one log: 2311774
Rows removed: 25865
Percentage removed: 1.106 %


In [68]:
ml_data = preprocessed_data.loc[valid_feature_rows].copy()

print("ML-ready rows:", len(ml_data))
print("Wells:", ml_data["WELL"].nunique())

ML-ready rows: 2311774
Wells: 118


In [69]:
print(
    ml_data[MODEL_FEATURES]
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

GR             0.71
RDEP_LOG10     4.00
RMED_LOG10     9.47
DTC           22.80
RHOB          40.54
dtype: float64


now we are checking whether in this cleaned row has the label or not😅

In [71]:
print("Missing labels:", ml_data[LABEL_CURVE].isna().sum())
print("Total rows:", len(ml_data))
print(
    "Percentage of missing labels:",
    round(ml_data[LABEL_CURVE].isna().mean() * 100, 3),
    "%"
)

Missing labels: 880800
Total rows: 2311774
Percentage of missing labels: 38.101 %


In [72]:
supervised_data = ml_data[
    ml_data[LABEL_CURVE].notna()
].copy()

print("Supervised rows:", len(supervised_data))
print("Wells:", supervised_data["WELL"].nunique())

Supervised rows: 1430974
Wells: 118


In [73]:
print("Missing labels:",
      supervised_data[LABEL_CURVE].isna().sum())

Missing labels: 0


supervised dataset

In [74]:
print("Rows:", len(supervised_data))
print("Wells:", supervised_data["WELL"].nunique())

print("\nMissing features:")
print(
    supervised_data[MODEL_FEATURES]
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

Rows: 1430974
Wells: 118

Missing features:
GR             0.09
RDEP_LOG10     0.77
RMED_LOG10     3.45
DTC            6.05
RHOB          13.14
dtype: float64


In [75]:
print(
    supervised_data[LABEL_CURVE]
    .value_counts()
    .sort_index()
)

FORCE_2020_LITHOFACIES_LITHOLOGY
30000.0    207671
65000.0    877042
65030.0    180446
70000.0     69498
70032.0     14043
74000.0      2391
80000.0     41038
86000.0      1807
88000.0     14712
90000.0      4754
93000.0       141
99000.0     17431
Name: count, dtype: int64


calculate the medians again using only the training wells AND labelled samples

In [76]:
train_mask = supervised_data["WELL"].isin(train_wells)

train_medians = (
    supervised_data.loc[train_mask, MODEL_FEATURES]
    .median()
)

print(train_medians)

GR             69.434273
RDEP_LOG10      0.170101
RMED_LOG10      0.165568
DTC           109.140079
RHOB            2.333478
dtype: float64


These are the values we'll use for the baseline imputation, and importantly, validation data will not influence them.

In [77]:
for feature in MODEL_FEATURES:
    supervised_data[f"{feature}_MISSING"] = (
        supervised_data[feature].isna().astype("int8")
    )

    supervised_data[feature] = (
        supervised_data[feature]
        .fillna(train_medians[feature])
    )

In [78]:
print(
    supervised_data[MODEL_FEATURES]
    .isna()
    .sum()
)

GR            0
RDEP_LOG10    0
RMED_LOG10    0
DTC           0
RHOB          0
dtype: int64


spliting the info...

In [81]:
split_map = split_df.set_index("WELL")["SPLIT"]

supervised_data["SPLIT"] = supervised_data["WELL"].map(split_map)


print(supervised_data["SPLIT"].value_counts())

SPLIT
train         1126021
validation     304953
Name: count, dtype: int64


now saving the dataset

In [82]:
OUTPUT_FILE = PROCESSED_DIR / "supervised_preprocessed.parquet"

supervised_data.to_parquet(
    OUTPUT_FILE,
    index=False
)

print("Saved:", OUTPUT_FILE)
print("Rows:", len(supervised_data))
print("Wells:", supervised_data["WELL"].nunique())

Saved: ../data/processed/supervised_preprocessed.parquet
Rows: 1430974
Wells: 118


save the preprocessing parameters:

In [83]:
import json

preprocessing_params = {
    "model_features": MODEL_FEATURES,
    "train_medians": train_medians.to_dict(),
    "log_transformed": [
        "RDEP",
        "RMED"
    ],
    "removed_if_all_features_missing": True,
    "invalid_rules": {
        "GR": "< 0 -> NaN",
        "DTC": "<= 0 -> NaN",
        "RHOB": "<= 0 -> NaN",
        "RDEP": "<= 0 -> NaN",
        "RMED": "<= 0 -> NaN"
    }
}

PARAM_FILE = PROCESSED_DIR / "preprocessing_params.json"

with open(PARAM_FILE, "w") as f:
    json.dump(preprocessing_params, f, indent=4)

print("Saved:", PARAM_FILE)

Saved: ../data/processed/preprocessing_params.json


In [84]:
print(supervised_data.shape)
print(supervised_data.columns.tolist())

(1430974, 15)
['GR', 'RDEP', 'RMED', 'DTC', 'RHOB', 'FORCE_2020_LITHOFACIES_LITHOLOGY', 'WELL', 'RDEP_LOG10', 'RMED_LOG10', 'GR_MISSING', 'RDEP_LOG10_MISSING', 'RMED_LOG10_MISSING', 'DTC_MISSING', 'RHOB_MISSING', 'SPLIT']


# 03 — Preprocessing Results

## Objective
Convert the raw FORCE 2020 well-log data into a consistent, reproducible dataset suitable for supervised lithology ML while preserving information about missing measurements and avoiding preprocessing leakage.

## Input
- **118 wells**
- Sampling interval: approximately **0.152 m**
- Candidate input curves: **GR, RDEP, RMED, DTC, RHOB**
- Target: `FORCE_2020_LITHOFACIES_LITHOLOGY`

## Preprocessing Steps & Results

### 1. Invalid-value handling
Based on `02_data_quality.ipynb`:
- `GR < 0` → `NaN`
- `DTC <= 0` → `NaN`
- `RHOB <= 0` → `NaN`
- `RDEP <= 0` → `NaN`
- `RMED <= 0` → `NaN`

Invalid measurements were marked as missing rather than replaced with arbitrary values.

### 2. Resistivity transformation
Because resistivity distributions were strongly skewed:
- `RDEP_LOG10 = log10(RDEP)`
- `RMED_LOG10 = log10(RMED)`

Final model features:
`GR, RDEP_LOG10, RMED_LOG10, DTC, RHOB`

### 3. Completely missing feature rows
Rows where all five model features were missing were removed:
- Total rows: **2,337,639**
- Removed: **25,865 (1.106%)**
- Remaining: **2,311,774**
- No wells were removed.

### 4. Unlabelled samples
Samples without lithology labels were excluded from supervised learning:
- Unlabelled samples: **880,800**
- Final labelled samples: **1,430,974**
- Wells represented: **118**

Unlabelled samples were retained conceptually for possible future inference and were not treated as unusable raw data.

### 5. Remaining feature missingness
After restricting to labelled samples:

| Feature | Missing |
|---|---:|
| GR | 0.09% |
| RDEP_LOG10 | 0.77% |
| RMED_LOG10 | 3.45% |
| DTC | 6.05% |
| RHOB | 13.14% |

Large continuous missing intervals were observed, particularly for DTC and RHOB. Maximum continuous gaps reached approximately **880 m (GR), 3459 m (RDEP), 3722 m (RMED), 3528 m (DTC), and 3990 m (RHOB)**. Therefore, blind interpolation across large gaps was rejected as a preprocessing strategy.

### 6. Missing-value baseline
A baseline strategy using **training-only median imputation + missingness indicators** was implemented.

Training medians:

| Feature | Median |
|---|---:|
| GR | 69.434273 |
| RDEP_LOG10 | 0.170101 |
| RMED_LOG10 | 0.165568 |
| DTC | 109.140079 |
| RHOB | 2.333478 |

Missing indicators were added for each model feature. Median values were calculated using training wells only to prevent preprocessing leakage.

### 7. Temporary development split
A temporary well-level split was used during preprocessing development:
- **94 training wells**
- **24 validation wells**
- **118 total wells**

The split is well-level to prevent samples from the same well appearing in both sets.

**Note:** This is a temporary development split. The official FORCE 2020 **98/10/10 train/public-test/blind-test allocation** will be incorporated before final model evaluation.

## Lithology Distribution

| Code | Samples |
|---:|---:|
| 30000 | 207,671 |
| 65000 | 877,042 |
| 65030 | 180,446 |
| 70000 | 69,498 |
| 70032 | 14,043 |
| 74000 | 2,391 |
| 80000 | 41,038 |
| 86000 | 1,807 |
| 88000 | 14,712 |
| 90000 | 4,754 |
| 93000 | 141 |
| 99000 | 17,431 |

The dataset is strongly imbalanced, with class **65000** dominant and **93000** extremely rare.

## Outputs
- `data/processed/supervised_preprocessed.parquet` — processed labelled ML dataset.
- `data/processed/preprocessing_params.json` — preprocessing rules, feature definitions, transformations, and training-derived medians.

## Final Result
`03_preprocessing.ipynb` produced a reproducible, leakage-aware supervised dataset containing **1,430,974 labelled samples from 118 wells**, with invalid values handled, resistivity transformed, completely feature-empty samples removed, missingness indicators preserved, and a training-only median-imputation baseline established.

**Next:** `04_feature_engineering.ipynb` will investigate additional geological/log-derived features that may improve lithology prediction.